[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Add ReID to Trackers

BoT-SORT can fuse visual ReID embeddings with IoU during association. In this
notebook you will enable appearance ReID with the [`reid`](https://reid.roboflow.com/latest/)
package, then compare BoT-SORT with and without embeddings on MOT17 val-half
using YOLOX detections and TrackEval metrics.

For threshold selection and MOT17 / SoccerNet results, see the
[ReID appearance guide](https://trackers.roboflow.com/latest/learn/reid/).

## Setup


### Check GPU availability

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do
that. In case of any problems navigate to `Runtime` -> `Change runtime type` ->
`Hardware accelerator`, set it to `GPU`, and then click `Save`.


In [ ]:
!nvidia-smi

### Install dependencies

You may see dependency conflict warnings in Google Colab. This is expected for the
preinstalled Google Colab environment and does not affect functionality.


In [ ]:
!pip install -q "trackers[reid]" matplotlib gdown


### Imports

Import Trackers, the `reid` package, and the MOT17 evaluation helpers used below.


In [ ]:
import shutil
import subprocess
import sys
import warnings
import zipfile
from pathlib import Path

import cv2
import gdown
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
import torch
from IPython.display import Video
from IPython.display import display as ipy_display
from reid import FASTREID_MOT17_SBS50, ReIDModel
from sklearn.decomposition import PCA

from trackers import BoTSORTTracker
from trackers.eval import evaluate_mot_sequences
from trackers.eval.box import box_iou
from trackers.eval.results import BenchmarkResult
from trackers.io.frames import load_mot_frame_image
from trackers.io.mot import _MOTOutput, load_mot_file

warnings.filterwarnings("ignore")

try:
    from google.colab import files

    IN_COLAB = True
    REPO_ROOT = Path("/content")
except ImportError:
    files = None
    IN_COLAB = False
    REPO_ROOT = Path("..").resolve()

VAL_SEQUENCES = [
    "MOT17-02-FRCNN",
    "MOT17-04-FRCNN",
    "MOT17-05-FRCNN",
    "MOT17-09-FRCNN",
    "MOT17-10-FRCNN",
    "MOT17-11-FRCNN",
    "MOT17-13-FRCNN",
]

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | {device}")

## Load ReID model

Load a MOT17 FastReID SBS50 encoder from the `reid` package. For other checkpoints
(for example OSNet on MSMT17), pass a different model id to
`ReIDModel.from_pretrained(...)`. See the
[`reid` training guide](https://reid.roboflow.com/latest/learn/train/).

In [ ]:
# Use another reid model id here to try OSNet or a custom checkpoint.
REID_ENCODER = FASTREID_MOT17_SBS50
REID_APPEARANCE_THRESHOLD = 0.2  # MOT17 re-ID study Table 8; BoT-SORT paper default 0.25
reid_model = ReIDModel.from_pretrained(REID_ENCODER)

print(f"Encoder: {REID_ENCODER}  |  appearance_threshold: {REID_APPEARANCE_THRESHOLD}")
print(reid_model.preprocessing.describe())


## Download MOT17 data

Download MOT17 val ground truth and frames with `trackers download`, then fetch the
YOLOX val detections used by the BoT-SORT / ByteTrack eval protocol. YOLOX frame IDs
are remapped to `1...N` so they align with MOT frame indexing.


In [ ]:
FORCE_DOWNLOAD = False

MOT17_VAL = REPO_ROOT / "mot17" / "val"
YOLOX_DIR = REPO_ROOT / "MOT17_yolox_dets"
YOLOX_VAL_DIR = YOLOX_DIR / "val"
YOLOX_ZIP = YOLOX_DIR / "yolox_detections_MOT17.zip"
YOLOX_GDRIVE_ID = "1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT"
OUTPUT_ROOT = REPO_ROOT / "trackers_reid_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def yolox_det_path(seq: str) -> Path:
    return YOLOX_VAL_DIR / f"{seq.replace('-FRCNN', '')}_val.txt"


def mot17_val_ready() -> bool:
    return all(
        (MOT17_VAL / seq / "gt" / "gt.txt").is_file() and (MOT17_VAL / seq / "img1").is_dir() for seq in VAL_SEQUENCES
    )


def yolox_ready() -> bool:
    return YOLOX_VAL_DIR.is_dir() and len(list(YOLOX_VAL_DIR.glob("MOT17-*_val.txt"))) >= len(VAL_SEQUENCES)


if FORCE_DOWNLOAD or not mot17_val_ready():
    subprocess.run(  # noqa: S603
        [
            sys.executable,
            "-m",
            "trackers.scripts",
            "download",
            "mot17",
            "--split",
            "val",
            "--asset",
            "annotations,frames",
            "-o",
            str(REPO_ROOT),
        ],
        check=True,
    )
else:
    print("MOT17 val already present.")

if FORCE_DOWNLOAD or not yolox_ready():
    YOLOX_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading YOLOX val detections...")
    gdown.download(id=YOLOX_GDRIVE_ID, output=str(YOLOX_ZIP), quiet=False)
    with zipfile.ZipFile(YOLOX_ZIP) as zf:
        zf.extractall(YOLOX_DIR)
else:
    print("YOLOX detections already present.")

SEQUENCE_PATHS: dict[str, dict] = {}
for seq in VAL_SEQUENCES:
    gt = MOT17_VAL / seq / "gt" / "gt.txt"
    img = MOT17_VAL / seq / "img1"
    det = yolox_det_path(seq)
    if not (gt.is_file() and img.is_dir() and det.is_file()):
        print(f"  skip {seq}: missing gt, img1, or YOLOX det")
        continue
    n_frames = len(list(img.glob("*.jpg")))
    SEQUENCE_PATHS[seq] = {"gt": gt, "img": img, "det": det, "n_frames": n_frames}
    print(f"  {seq}: {n_frames} frames")

ACTIVE_SEQUENCES = list(SEQUENCE_PATHS)
if not ACTIVE_SEQUENCES:
    raise RuntimeError("No sequences ready - re-run downloads above.")

SEQMAP_PATH = OUTPUT_ROOT / "MOT17-val.txt"
SEQMAP_PATH.write_text("name\n" + "\n".join(ACTIVE_SEQUENCES) + "\n")
print(f"\n{len(ACTIVE_SEQUENCES)} sequences -> outputs in {OUTPUT_ROOT}")

## Tracking helpers

Helpers for loading YOLOX detections, running a tracker over a sequence, and writing
MOT predictions. Set `RERUN[name]=False` to reuse cached predictions under
`trackers_reid_outputs/`.


In [ ]:
RERUN = {
    "botsort_baseline": True,
    "botsort_reid": True,
}


def _yolox_frame_offset(det_path: Path) -> int:
    min_frame = None
    with det_path.open() as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame = int(float(parts[0]))
            min_frame = frame if min_frame is None else min(min_frame, frame)
    return (min_frame - 1) if min_frame and min_frame > 1 else 0


def load_yolox_dets(det_path: Path) -> dict[int, sv.Detections]:
    offset = _yolox_frame_offset(det_path)
    by_frame: dict[int, list[list[float]]] = {}
    with det_path.open() as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame = int(float(parts[0])) - offset
            if frame < 1:
                continue
            x1, y1, x2, y2, score = map(float, parts[1:6])
            if score <= 0:
                continue
            by_frame.setdefault(frame, []).append([x1, y1, x2, y2, score])
    return {
        frame: sv.Detections(
            xyxy=np.array(boxes, dtype=np.float32)[:, :4],
            confidence=np.array(boxes, dtype=np.float32)[:, 4],
        )
        for frame, boxes in by_frame.items()
    }


def fmt_metrics(result: BenchmarkResult) -> tuple[float, float, float, float, float, int]:
    a = result.aggregate
    return (
        (a.HOTA.HOTA * 100 if a.HOTA else float("nan")),
        (a.HOTA.AssA * 100 if a.HOTA else float("nan")),
        (a.HOTA.DetA * 100 if a.HOTA else float("nan")),
        (a.CLEAR.MOTA * 100 if a.CLEAR else float("nan")),
        (a.Identity.IDF1 * 100 if a.Identity else float("nan")),
        (a.CLEAR.IDSW if a.CLEAR else 0),
    )


def print_metrics(label: str, result: BenchmarkResult) -> None:
    hota, _assa, _deta, mota, idf1, idsw = fmt_metrics(result)
    print(f"{label}: HOTA {hota:6.2f}  MOTA {mota:6.2f}  IDF1 {idf1:6.2f}  IDSW {idsw}")


def run_tracking(name: str, factory, *, use_frames: bool) -> Path:
    pred_dir = OUTPUT_ROOT / name / "preds"
    pred_dir.mkdir(parents=True, exist_ok=True)

    for seq in ACTIVE_SEQUENCES:
        spec = SEQUENCE_PATHS[seq]
        dets = load_yolox_dets(spec["det"])
        images = sorted(spec["img"].glob("*.jpg"))
        tracker = factory()

        with _MOTOutput(pred_dir / f"{seq}.txt") as out:
            for frame_idx in range(1, spec["n_frames"] + 1):
                frame = None
                if use_frames and frame_idx <= len(images):
                    frame = cv2.imread(str(images[frame_idx - 1]))
                tracked = tracker.update(dets.get(frame_idx, sv.Detections.empty()), frame)
                if tracked.tracker_id is not None:
                    tracked = tracked[tracked.tracker_id != -1]
                out.write(frame_idx, tracked)
        print(f"  {seq}: {spec['n_frames']} frames")

    return pred_dir


def evaluate(name: str, pred_dir: Path) -> BenchmarkResult:
    result = evaluate_mot_sequences(
        gt_dir=MOT17_VAL,
        tracker_dir=pred_dir,
        seqmap=SEQMAP_PATH,
        metrics=["CLEAR", "HOTA", "Identity"],
    )
    cache = OUTPUT_ROOT / name / "eval_results.json"
    cache.parent.mkdir(parents=True, exist_ok=True)
    result.save(cache)
    return result


def load_or_run(name: str, factory, *, use_frames: bool) -> BenchmarkResult:
    pred_dir = OUTPUT_ROOT / name / "preds"
    cache = OUTPUT_ROOT / name / "eval_results.json"
    preds_ok = pred_dir.exists() and all((pred_dir / f"{s}.txt").exists() for s in ACTIVE_SEQUENCES)

    ran = False
    if RERUN.get(name, True) or not preds_ok:
        print(f"Running {name}...")
        pred_dir = run_tracking(name, factory, use_frames=use_frames)
        ran = True
    else:
        print(f"Using cached preds: {pred_dir}")

    if not ran and cache.exists():
        print(f"Using cached eval: {cache}")
        return BenchmarkResult.load(cache)

    print(f"Evaluating {name}...")
    return evaluate(name, pred_dir)


def match_dets_to_gt(gt_frame, det_xyxy: np.ndarray, min_iou: float = 0.5) -> np.ndarray:
    if len(det_xyxy) == 0:
        return np.array([], dtype=np.int64)
    gt_xyxy = sv.xywh_to_xyxy(gt_frame.boxes)
    keep = (gt_frame.confidences > 0) & (gt_frame.classes == 1)
    gt_xyxy, gt_ids = gt_xyxy[keep], gt_frame.ids[keep]
    if len(gt_xyxy) == 0:
        return np.full(len(det_xyxy), -1, dtype=np.int64)
    ious = box_iou(det_xyxy.astype(np.float64), gt_xyxy.astype(np.float64))
    out = np.full(len(det_xyxy), -1, dtype=np.int64)
    for i in range(len(det_xyxy)):
        j = int(np.argmax(ious[i]))
        if ious[i, j] >= min_iou:
            out[i] = int(gt_ids[j])
    return out

## Run BoT-SORT with and without ReID

Run the baseline BoT-SORT tracker (CMC on, no appearance) and the same tracker with
a `reid_model`. When ReID is enabled, pass the current frame to `update()` so
embeddings can be extracted.


In [ ]:
EXPERIMENTS = [
    (
        "botsort_baseline",
        "BoT-SORT (baseline)",
        lambda: BoTSORTTracker(enable_cmc=True),
        True,
    ),
    (
        "botsort_reid",
        "BoT-SORT + ReID",
        lambda: BoTSORTTracker(
            enable_cmc=True,
            reid_model=reid_model,
            reid_ema_alpha=0.9,
            appearance_threshold=REID_APPEARANCE_THRESHOLD,
        ),
        True,
    ),
]

results: dict[str, BenchmarkResult] = {}
for name, label, factory, use_frames in EXPERIMENTS:
    results[name] = load_or_run(name, factory, use_frames=use_frames)
    print_metrics(label, results[name])
    print()

result_baseline = results["botsort_baseline"]
result_reid = results["botsort_reid"]

## ReID embedding visualization (optional)

PCA of track embeddings on one sequence. This is a quick sanity check that
same-ID crops cluster and different IDs separate in embedding space.


In [ ]:
VIZ_SEQ = "MOT17-02-FRCNN"
VIZ_STRIDE, VIZ_MAX_FRAMES, VIZ_MAX_POINTS, VIZ_MAX_CROPS = 5, 40, 300, 24

spec = SEQUENCE_PATHS[VIZ_SEQ]
gt_by_frame = load_mot_file(spec["gt"])
dets_by_frame = load_yolox_dets(spec["det"])
images = sorted(spec["img"].glob("*.jpg"))

crops, embeddings, gt_ids = [], [], []
for frame_idx in list(range(1, spec["n_frames"] + 1, VIZ_STRIDE))[:VIZ_MAX_FRAMES]:
    dets = dets_by_frame.get(frame_idx)
    gt = gt_by_frame.get(frame_idx)
    if dets is None or gt is None or len(dets) == 0:
        continue
    dets = dets[dets.confidence >= 0.5]
    if len(dets) == 0:
        continue
    bgr = cv2.imread(str(images[frame_idx - 1]))
    if bgr is None:
        continue
    matched = match_dets_to_gt(gt, dets.xyxy)
    feats = reid_model.extract_features(dets, bgr)
    for i in range(len(dets)):
        if matched[i] < 0:
            continue
        crop = sv.crop_image(bgr, dets.xyxy[i].astype(int))
        if crop.size == 0:
            continue
        crops.append(crop[:, :, ::-1])
        embeddings.append(feats[i])
        gt_ids.append(int(matched[i]))

if not embeddings:
    raise RuntimeError("No matched crops - try another sequence or lower confidence threshold")

emb = np.stack(embeddings)
labels = np.array(gt_ids)
if len(emb) > VIZ_MAX_POINTS:
    idx = np.linspace(0, len(emb) - 1, VIZ_MAX_POINTS, dtype=int)
    emb, labels, crops = emb[idx], labels[idx], [crops[i] for i in idx]

coords = PCA(n_components=2, random_state=0).fit_transform(emb)
unique = np.unique(labels)
colors = {pid: plt.colormaps["tab20"](i % 20) for i, pid in enumerate(unique)}

fig, (ax_pca, ax_crop) = plt.subplots(1, 2, figsize=(14, 6))
for pid in unique:
    m = labels == pid
    ax_pca.scatter(coords[m, 0], coords[m, 1], s=28, alpha=0.85, color=colors[pid], label=f"id {pid}")
ax_pca.set(title=f"{VIZ_SEQ} - PCA by GT id", xlabel="PC1", ylabel="PC2")
ax_pca.grid(True, alpha=0.3)
if len(unique) <= 12:
    ax_pca.legend(fontsize=8)

n_show = min(len(crops), VIZ_MAX_CROPS)
ncols, nrows = 6, int(np.ceil(n_show / 6))
mosaic = np.full((nrows * 64, ncols * 32, 3), 255, dtype=np.uint8)
for k in range(n_show):
    r, c = divmod(k, ncols)
    tile = cv2.resize(crops[k], (32, 64))
    y, x = r * 64, c * 32
    mosaic[y : y + 64, x : x + 32] = tile
    rgb = (np.array(colors[labels[k]])[:3] * 255).astype(np.uint8)
    mosaic[y : y + 2, x : x + 32] = rgb
    mosaic[y + 62 : y + 64, x : x + 32] = rgb

ax_crop.imshow(mosaic)
ax_crop.set(title=f"Sample crops ({n_show})")
ax_crop.axis("off")
plt.tight_layout()
plt.show()
print(f"{len(coords)} points, {len(unique)} GT ids")

## Appearance distance histogram

Encoder diagnostic on MOT17 val **GT crops** (no detector). Pair sampling matches
what BoT-SORT association can see: one video at a time, within the lost-track
horizon.

**Protocol**
- Axis: `d_app = 0.5 * (1 - cos)` (BoT-SORT `embedding_distance / 2`)
- Y-axis: probability (weights `1/n`)
- GT: eval pedestrians (`conf > 0`, `class == 1`)
- Positives: same ID, same sequence, `1 <= |frame gap| <= MAX_FRAME_GAP`
- Negatives: different ID, same sequence, `1 <= |frame gap| <= MAX_FRAME_GAP`
- Sampling: equal quota per sequence, and same-ID pairs pick an identity uniformly
  so that long tracks and crowded sequences do not dominate either curve
- θ lines: 0.25 (default) and `REID_APPEARANCE_THRESHOLD`

`MAX_FRAME_GAP` defaults to 30 (BoT-SORT `lost_track_buffer` at 30 FPS). Set
`SAVE_DOCS_ASSET = True` to write
`docs/assets/reid/mot17-fastreid-appearance-distances.png`.


In [ ]:
from collections import defaultdict

N_INTRA = 5000
N_INTER = 10000
MIN_FRAME_GAP = 1
MAX_FRAME_GAP = 30  # ~lost_track_buffer @ 30 FPS
CANDIDATE_THETAS = (0.20, 0.25)
_DAPP_BINS = np.linspace(0.0, 1.0, 51)
SAVE_DOCS_ASSET = False
DOCS_ASSET_PATH = REPO_ROOT / "docs" / "assets" / "reid" / "mot17-fastreid-appearance-distances.png"


def load_eval_pedestrian_gt(gt_path: Path):
    """Eval pedestrians (conf>0, class==1). No visibility / min-size filters."""
    by_frame: dict[int, list[tuple[int, np.ndarray]]] = defaultdict(list)
    with gt_path.open() as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 8:
                continue
            frame = int(float(parts[0]))
            tid = int(float(parts[1]))
            x, y, w, h = map(float, parts[2:6])
            conf, cls = float(parts[6]), int(float(parts[7]))
            if conf <= 0 or cls != 1:
                continue
            by_frame[frame].append((tid, np.array([x, y, x + w, y + h], dtype=np.float32)))
    return by_frame


def collect_gt_embeddings(
    model: ReIDModel,
    sequences: list[str],
    *,
    frame_stride: int = 1,
    max_frames_per_seq: int | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Return embeddings, labels (seq_tid), frame ids, sequence ids."""
    embeddings: list[np.ndarray] = []
    labels: list[int] = []
    frame_ids: list[int] = []
    seq_ids: list[int] = []
    label_by_key: dict[str, int] = {}
    seq_by_name: dict[str, int] = {}
    for seq in sequences:
        if seq not in seq_by_name:
            seq_by_name[seq] = len(seq_by_name)
        sid = seq_by_name[seq]
        spec = SEQUENCE_PATHS[seq]
        gt_by_frame = load_eval_pedestrian_gt(spec["gt"])
        images = sorted(spec["img"].glob("*.jpg"))
        frame_indices = list(range(1, spec["n_frames"] + 1, frame_stride))
        if max_frames_per_seq is not None:
            frame_indices = frame_indices[:max_frames_per_seq]
        for frame_idx in frame_indices:
            rows = gt_by_frame.get(frame_idx)
            if not rows:
                continue
            bgr = cv2.imread(str(images[frame_idx - 1]))
            if bgr is None:
                continue
            xyxy = np.stack([r[1] for r in rows], axis=0)
            track_ids = np.asarray([r[0] for r in rows], dtype=np.int64)
            feats = model.extract_features(sv.Detections(xyxy=xyxy), bgr)
            for i, tid in enumerate(track_ids):
                key = f"{seq}_{int(tid)}"
                if key not in label_by_key:
                    label_by_key[key] = len(label_by_key)
                embeddings.append(feats[i])
                labels.append(label_by_key[key])
                frame_ids.append(frame_idx)
                seq_ids.append(sid)
    if not embeddings:
        raise RuntimeError("No GT embeddings (conf>0, class==1 pedestrians).")
    return (
        np.stack(embeddings),
        np.asarray(labels, dtype=np.int64),
        np.asarray(frame_ids, dtype=np.int64),
        np.asarray(seq_ids, dtype=np.int64),
    )


def _d_app(a: np.ndarray, b: np.ndarray) -> float:
    return 0.5 * (1.0 - float(a @ b))


def sample_association_local_distances(
    embeddings: np.ndarray,
    gt_ids: np.ndarray,
    *,
    frame_ids: np.ndarray,
    seq_ids: np.ndarray,
    n_intra: int = N_INTRA,
    n_inter: int = N_INTER,
    min_frame_gap: int = MIN_FRAME_GAP,
    max_frame_gap: int = MAX_FRAME_GAP,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """Same-video pairs within max_frame_gap (tracker association horizon).

    Pairs are drawn directly rather than enumerated into a pool, and every sequence
    gets the same quota, so no single crowded sequence can decide the histogram.
    Same-ID pairs pick an identity uniformly so long tracks do not dominate.
    """
    if min_frame_gap < 1:
        raise ValueError("min_frame_gap must be >= 1, otherwise a crop can pair with itself")
    normed = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)
    rng = np.random.default_rng(seed)

    # Per sequence: crop indexes sorted by frame, their frames and ids, and the
    # slot lists of every identity seen more than once.
    per_seq: dict[int, tuple] = {}
    for sid in np.unique(seq_ids):
        slots = np.flatnonzero(seq_ids == sid)
        slots = slots[np.argsort(frame_ids[slots], kind="stable")]
        by_id: dict[int, list[int]] = defaultdict(list)
        for pos, idx in enumerate(slots):
            by_id[int(gt_ids[idx])].append(pos)
        tracks = [np.asarray(v) for v in by_id.values() if len(v) > 1]
        per_seq[int(sid)] = (slots, frame_ids[slots], gt_ids[slots], tracks)

    def pick_partner(frames: np.ndarray, anchor_frame: int) -> int | None:
        """Uniform slot of ``frames`` whose gap to ``anchor_frame`` is inside the band."""
        before_lo = int(np.searchsorted(frames, anchor_frame - max_frame_gap, "left"))
        before_hi = int(np.searchsorted(frames, anchor_frame - min_frame_gap, "right"))
        after_lo = int(np.searchsorted(frames, anchor_frame + min_frame_gap, "left"))
        after_hi = int(np.searchsorted(frames, anchor_frame + max_frame_gap, "right"))
        n_before, n_after = max(0, before_hi - before_lo), max(0, after_hi - after_lo)
        if n_before + n_after == 0:
            return None
        draw = int(rng.integers(n_before + n_after))
        return before_lo + draw if draw < n_before else after_lo + (draw - n_before)

    def draw_pair(sid: int, same_id: bool) -> tuple[int, int] | None:
        slots, frames, ids, tracks = per_seq[sid]
        if same_id:
            if not tracks:
                return None
            track = tracks[int(rng.integers(len(tracks)))]
            anchor = int(track[int(rng.integers(len(track)))])
            partner = pick_partner(frames[track], int(frames[anchor]))
            partner = None if partner is None else int(track[partner])
        else:
            anchor = int(rng.integers(len(slots)))
            partner = pick_partner(frames, int(frames[anchor]))
            if partner is not None and ids[partner] == ids[anchor]:
                partner = None
        if partner is None or partner == anchor:
            return None
        return int(slots[anchor]), int(slots[partner])

    sampled: list[np.ndarray] = []
    sids = sorted(per_seq)
    for quota, same_id in ((n_intra, True), (n_inter, False)):
        distances: list[float] = []
        for k, sid in enumerate(sids):
            wanted = quota // len(sids) + (1 if k < quota % len(sids) else 0)
            drawn = 0
            for _ in range(wanted * 64):
                if drawn >= wanted:
                    break
                pair = draw_pair(sid, same_id)
                if pair is None:
                    continue
                distances.append(_d_app(normed[pair[0]], normed[pair[1]]))
                drawn += 1
        if not distances:
            kind = "same-ID" if same_id else "different-ID"
            raise ValueError(f"No {kind} pairs with {min_frame_gap}<=|Delta frame|<={max_frame_gap}.")
        sampled.append(np.asarray(distances))
    return sampled[0], sampled[1]


emb_gt, gt_ids, frame_ids, seq_ids = collect_gt_embeddings(
    reid_model,
    ACTIVE_SEQUENCES,
    frame_stride=1,
    max_frames_per_seq=None,
)
print(f"GT pool: {len(emb_gt)} crops, {len(np.unique(gt_ids))} ids, {len(np.unique(seq_ids))} sequences")

intra, inter = sample_association_local_distances(
    emb_gt,
    gt_ids,
    frame_ids=frame_ids,
    seq_ids=seq_ids,
    n_intra=N_INTRA,
    n_inter=N_INTER,
    min_frame_gap=MIN_FRAME_GAP,
    max_frame_gap=MAX_FRAME_GAP,
    seed=0,
)
print(f"pairs: same-ID={len(intra)}  diff-ID={len(inter)}  (same seq, |Delta frame|<={MAX_FRAME_GAP})")
print(
    f"d_app means: same-ID={intra.mean():.3f}  diff-ID={inter.mean():.3f}  "
    f"gap={inter.mean() - intra.mean():.3f}  "
    f"same-ID p95={np.quantile(intra, 0.95):.3f}"
)

fig, ax = plt.subplots(figsize=(8, 4.5))
w_intra = np.full(len(intra), 1.0 / len(intra))
w_inter = np.full(len(inter), 1.0 / len(inter))
ax.hist(
    intra,
    bins=_DAPP_BINS,
    weights=w_intra,
    alpha=0.65,
    label=f"same-ID (n={len(intra)})",
    color="#3366CC",
)
ax.hist(
    inter,
    bins=_DAPP_BINS,
    weights=w_inter,
    alpha=0.65,
    label=f"different-ID (n={len(inter)})",
    color="#DC3912",
)
ax.axvline(0.25, color="#666666", ls=":", lw=1.5, label="θ=0.25 (default)")
ax.axvline(
    REID_APPEARANCE_THRESHOLD,
    color="#111111",
    ls="--",
    lw=1.8,
    label=f"θ={REID_APPEARANCE_THRESHOLD:.2f} (selected)",
)
ax.set(
    xlabel=r"$0.5\cdot$ cosine distance",
    ylabel="probability",
    title=f"{REID_ENCODER} on MOT17 val GT",
    xlim=(0.0, 0.6),
)
ax.legend(frameon=False, fontsize=9)
ax.grid(True, alpha=0.25)
fig.tight_layout()
if SAVE_DOCS_ASSET:
    DOCS_ASSET_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(DOCS_ASSET_PATH, dpi=150, bbox_inches="tight")
    print(f"Wrote {DOCS_ASSET_PATH}")
plt.show()

print(f"{'θ':>6}  {'same-ID < θ':>12}  {'diff-ID < θ':>12}")
for theta in CANDIDATE_THETAS:
    same_below = 100 * float(np.mean(intra < theta))
    diff_below = 100 * float(np.mean(inter < theta))
    note = ""
    if abs(theta - REID_APPEARANCE_THRESHOLD) < 1e-9:
        note = "  <- selected"
    elif abs(theta - 0.25) < 1e-9:
        note = "  <- paper"
    print(f"{theta:6.2f}  {same_below:11.1f}%  {diff_below:11.1f}%{note}")

## How far the threshold carries

The histogram above fixes the frame gap at `MAX_FRAME_GAP`, so it only describes
re-association over that horizon. Sweeping the gap shows how long a track can stay
lost before appearance stops helping to re-find it.

ROC AUC is used as the summary because it needs no operating point, so the curve
does not depend on a chosen true-positive or false-positive rate. The printed rates
evaluate the θ you already picked rather than deriving a new one.

In [ ]:
GAP_BUCKETS = [(1, 1), (2, 5), (6, 15), (16, 30), (31, 60), (61, 120), (121, 240)]
N_SWEEP_PER_CLASS = 4000
DOCS_SWEEP_ASSET_PATH = REPO_ROOT / "docs" / "assets" / "reid" / "mot17-fastreid-appearance-distances-vs-gap.png"


def roc_auc(intra: np.ndarray, inter: np.ndarray) -> float:
    """P(same-ID distance < different-ID distance), ties counted as half."""
    inter_sorted = np.sort(inter)
    right = np.searchsorted(inter_sorted, intra, side="right")
    left = np.searchsorted(inter_sorted, intra, side="left")
    return float(np.mean(((len(inter) - right) + 0.5 * (right - left)) / len(inter)))


sweep: list[tuple[str, np.ndarray, np.ndarray]] = []
for lo, hi in GAP_BUCKETS:
    try:
        gap_intra, gap_inter = sample_association_local_distances(
            emb_gt,
            gt_ids,
            frame_ids=frame_ids,
            seq_ids=seq_ids,
            n_intra=N_SWEEP_PER_CLASS,
            n_inter=N_SWEEP_PER_CLASS,
            min_frame_gap=lo,
            max_frame_gap=hi,
        )
    except ValueError:
        print(f"gap {lo}-{hi}: no pairs, skipped")
        continue
    sweep.append((str(lo) if lo == hi else f"{lo}-{hi}", gap_intra, gap_inter))

x = np.arange(len(sweep))
intra_q = np.array([np.percentile(row[1], [25, 50, 95]) for row in sweep])
inter_q = np.array([np.percentile(row[2], [5, 50, 75]) for row in sweep])

fig, (ax_dist, ax_auc) = plt.subplots(
    2, 1, figsize=(8, 6.5), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.0]}
)
ax_dist.fill_between(x, intra_q[:, 0], intra_q[:, 2], color="#3366CC", alpha=0.25, label="same-ID p25-p95")
ax_dist.plot(x, intra_q[:, 1], color="#3366CC", marker="o", lw=2, label="same-ID median")
ax_dist.fill_between(x, inter_q[:, 0], inter_q[:, 2], color="#DC3912", alpha=0.25, label="diff-ID p5-p75")
ax_dist.plot(x, inter_q[:, 1], color="#DC3912", marker="o", lw=2, label="diff-ID median")
ax_dist.axhline(0.25, color="#666666", ls=":", lw=1.5, label="θ=0.25 (default)")
ax_dist.axhline(
    REID_APPEARANCE_THRESHOLD,
    color="#111111",
    ls="--",
    lw=1.5,
    label=f"θ={REID_APPEARANCE_THRESHOLD:.2f} (selected)",
)
ax_dist.set(ylabel=r"$d_{app} = 0.5\cdot(1-\cos)$", title=f"{REID_ENCODER}: separability vs frame gap")
ax_dist.legend(loc="lower right", fontsize=8, ncol=2, framealpha=0.92, edgecolor="none")
ax_dist.grid(True, alpha=0.25)

ax_auc.plot(x, [roc_auc(row[1], row[2]) for row in sweep], color="#111111", marker="s", lw=2)
ax_auc.axhline(0.5, color="#999999", ls=":", lw=1.2)
ax_auc.set(
    xlabel="frame gap between the two crops",
    ylabel="ROC AUC",
    ylim=(0.45, 1.02),
    xticks=x,
    xticklabels=[row[0] for row in sweep],
)
ax_auc.grid(True, alpha=0.25)
fig.tight_layout()
if SAVE_DOCS_ASSET:
    DOCS_SWEEP_ASSET_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(DOCS_SWEEP_ASSET_PATH, dpi=150, bbox_inches="tight")
    print(f"Wrote {DOCS_SWEEP_ASSET_PATH}")
plt.show()

print(f"{'gap':>10}  {'AUC':>6}  {'same-ID < θ':>12}  {'diff-ID < θ':>12}")
for label, gap_intra, gap_inter in sweep:
    print(
        f"{label:>10}  {roc_auc(gap_intra, gap_inter):6.3f}  "
        f"{100 * np.mean(gap_intra < REID_APPEARANCE_THRESHOLD):11.1f}%  "
        f"{100 * np.mean(gap_inter < REID_APPEARANCE_THRESHOLD):11.1f}%"
    )

## Results

Compare your BoT-SORT baseline and +ReID runs against published MOT17 val
references.


### Reference targets

**Primary:** [*Does Re-ID Really Help in Multi-Object Tracking?*](https://www-sop.inria.fr/members/Francois.Bremond/Postscript/Tomasz__SCCAI_2025.pdf) (2025). BoT-SORT + YOLOX + MOT17 FastReID, app th=0.2. Combined val scores from **Table 8 (HOTA)** and **Table 13 (IDF1)**; MOTA is not reported for this YOLOX setup.

| Config | HOTA | IDF1 |
|---|---:|---:|
| No re-ID | 68.43 | 80.92 |
| MOT17 FastReID, app th=0.2 | 68.95 | 81.98 |
| **ReID improvement (reference)** | **+0.52** | **+1.06** |

**Secondary:** [BoT-SORT paper](https://arxiv.org/abs/2206.14651) (Table 1, MOT17 val):

| Method | HOTA | MOTA | IDF1 |
|---|---:|---:|---:|
| BoT-SORT | 69.11 | 78.39 | 81.53 |
| BoT-SORT + ReID | 69.17 | 78.46 | 82.07 |
| **ReID improvement (BoT-SORT paper)** | **+0.06** | **+0.07** | **+0.54** |


In [ ]:
# MOT17 re-ID study reference - Table 8 (HOTA) + Table 13 (IDF1), COMBINED row.
# MOTA is not reported for the YOLOX setup in that study.
REID_STUDY_NO_REID = {"hota": 68.428, "mota": None, "idf1": 80.92}
REID_STUDY_MOT17_TH02 = {"hota": 68.951, "mota": None, "idf1": 81.984}
REID_STUDY_REID_DELTA = {k: REID_STUDY_MOT17_TH02[k] - REID_STUDY_NO_REID[k] for k in ("hota", "idf1")}

# BoT-SORT paper Table 1 (MOT17 val, YOLOX).
BOTSORT_PAPER = {"hota": 69.11, "mota": 78.39, "idf1": 81.53}
BOTSORT_PAPER_REID = {"hota": 69.17, "mota": 78.46, "idf1": 82.07}
BOTSORT_PAPER_REID_DELTA = {k: BOTSORT_PAPER_REID[k] - BOTSORT_PAPER[k] for k in BOTSORT_PAPER}


def fmt_ref_metric(value: float | None) -> str:
    return f"{value:6.2f}" if value is not None else "     -"


def seq_metrics(result: BenchmarkResult, seq: str) -> tuple[float, float, float, int]:
    s = result.sequences.get(seq)
    if s is None:
        return float("nan"), float("nan"), float("nan"), 0
    return (
        s.HOTA.HOTA * 100 if s.HOTA else float("nan"),
        s.HOTA.AssA * 100 if s.HOTA else float("nan"),
        s.Identity.IDF1 * 100 if s.Identity else float("nan"),
        s.CLEAR.IDSW if s.CLEAR else 0,
    )


botsort_rows = [
    ("BoT-SORT (baseline)", result_baseline),
    ("BoT-SORT + ReID", result_reid),
]

print("BoT-SORT - trackers (aggregate, all val sequences)")
print(f"{'Config':<28}  {'HOTA':>6}  {'AssA':>6}  {'DetA':>6}  {'MOTA':>6}  {'IDF1':>6}  {'IDSW':>5}")
print("-" * 72)
for label, res in botsort_rows:
    hota, assa, deta, mota, idf1, idsw = fmt_metrics(res)
    print(f"{label:<28}  {hota:6.2f}  {assa:6.2f}  {deta:6.2f}  {mota:6.2f}  {idf1:6.2f}  {idsw:5d}")

b = fmt_metrics(result_baseline)
r = fmt_metrics(result_reid)
print(
    f"\nBoT-SORT ReID uplift (trackers): "
    f"delta HOTA {r[0] - b[0]:+6.2f}  delta MOTA {r[3] - b[3]:+6.2f}  "
    f"delta IDF1 {r[4] - b[4]:+6.2f}  delta IDSW {int(r[5] - b[5]):+5d}"
)

print("\nBoT-SORT vs MOT17 re-ID study (primary - Table 8 + Table 13)")
print(f"{'':28}  {'HOTA':>6}  {'MOTA':>6}  {'IDF1':>6}")
print("-" * 52)
print(
    f"{'Reference (no re-ID)':<28}  "
    f"{REID_STUDY_NO_REID['hota']:6.2f}  {fmt_ref_metric(REID_STUDY_NO_REID['mota'])}  "
    f"{REID_STUDY_NO_REID['idf1']:6.2f}"
)
print(
    f"{'trackers (baseline)':<28}  {b[0]:6.2f}  {b[3]:6.2f}  {b[4]:6.2f}  "
    f"  d {b[0] - REID_STUDY_NO_REID['hota']:+5.2f}  "
    f"{'-':>6}  {b[4] - REID_STUDY_NO_REID['idf1']:+5.2f}"
)
print(
    f"{'Reference (MOT17 th=0.2)':<28}  "
    f"{REID_STUDY_MOT17_TH02['hota']:6.2f}  {fmt_ref_metric(REID_STUDY_MOT17_TH02['mota'])}  "
    f"{REID_STUDY_MOT17_TH02['idf1']:6.2f}"
)
print(
    f"{'trackers (+ ReID)':<28}  {r[0]:6.2f}  {r[3]:6.2f}  {r[4]:6.2f}  "
    f"  d {r[0] - REID_STUDY_MOT17_TH02['hota']:+5.2f}  "
    f"{'-':>6}  {r[4] - REID_STUDY_MOT17_TH02['idf1']:+5.2f}"
)
print(
    f"\nReID uplift vs reference study\n"
    f"  delta HOTA  trackers {r[0] - b[0]:+6.2f}   reference {REID_STUDY_REID_DELTA['hota']:+6.2f}   "
    f"gap {(r[0] - b[0]) - REID_STUDY_REID_DELTA['hota']:+6.2f}\n"
    f"  delta MOTA  trackers {r[3] - b[3]:+6.2f}   reference      -\n"
    f"  delta IDF1  trackers {r[4] - b[4]:+6.2f}   reference {REID_STUDY_REID_DELTA['idf1']:+6.2f}   "
    f"gap {(r[4] - b[4]) - REID_STUDY_REID_DELTA['idf1']:+6.2f}"
)

print("\nBoT-SORT vs BoT-SORT paper Table 1 (secondary)")
print(f"{'':28}  {'HOTA':>6}  {'MOTA':>6}  {'IDF1':>6}")
print("-" * 52)
print(
    f"{'BoT-SORT paper':<28}  {BOTSORT_PAPER['hota']:6.2f}  {BOTSORT_PAPER['mota']:6.2f}  {BOTSORT_PAPER['idf1']:6.2f}"
)
print(
    f"{'trackers (baseline)':<28}  {b[0]:6.2f}  {b[3]:6.2f}  {b[4]:6.2f}  "
    f"  d {b[0] - BOTSORT_PAPER['hota']:+5.2f}  "
    f"{b[3] - BOTSORT_PAPER['mota']:+5.2f}  {b[4] - BOTSORT_PAPER['idf1']:+5.2f}"
)
print(
    f"{'BoT-SORT paper + ReID':<28}  {BOTSORT_PAPER_REID['hota']:6.2f}  "
    f"{BOTSORT_PAPER_REID['mota']:6.2f}  {BOTSORT_PAPER_REID['idf1']:6.2f}"
)
print(
    f"{'trackers (+ ReID)':<28}  {r[0]:6.2f}  {r[3]:6.2f}  {r[4]:6.2f}  "
    f"  d {r[0] - BOTSORT_PAPER_REID['hota']:+5.2f}  "
    f"{r[3] - BOTSORT_PAPER_REID['mota']:+5.2f}  {r[4] - BOTSORT_PAPER_REID['idf1']:+5.2f}"
)
print(
    f"\nReID uplift vs BoT-SORT paper\n"
    f"  delta HOTA  trackers {r[0] - b[0]:+6.2f}   BoT-SORT paper {BOTSORT_PAPER_REID_DELTA['hota']:+6.2f}   "
    f"gap {(r[0] - b[0]) - BOTSORT_PAPER_REID_DELTA['hota']:+6.2f}\n"
    f"  delta MOTA  trackers {r[3] - b[3]:+6.2f}   BoT-SORT paper {BOTSORT_PAPER_REID_DELTA['mota']:+6.2f}   "
    f"gap {(r[3] - b[3]) - BOTSORT_PAPER_REID_DELTA['mota']:+6.2f}\n"
    f"  delta IDF1  trackers {r[4] - b[4]:+6.2f}   BoT-SORT paper {BOTSORT_PAPER_REID_DELTA['idf1']:+6.2f}   "
    f"gap {(r[4] - b[4]) - BOTSORT_PAPER_REID_DELTA['idf1']:+6.2f}"
)

### Per-sequence comparison

Break down HOTA and IDF1 per MOT17 val sequence against the MOT17 re-ID study
Tables 8 and 13.


In [ ]:
REID_STUDY_PER_SEQ = {
    "MOT17-02": {
        "no_reid": {"hota": 47.131, "idf1": 56.968},
        "mot17_th02": {"hota": 49.304, "idf1": 60.0},
    },
    "MOT17-04": {
        "no_reid": {"hota": 78.976, "idf1": 91.021},
        "mot17_th02": {"hota": 79.046, "idf1": 90.864},
    },
    "MOT17-05": {
        "no_reid": {"hota": 60.078, "idf1": 75.124},
        "mot17_th02": {"hota": 61.469, "idf1": 77.969},
    },
    "MOT17-09": {
        "no_reid": {"hota": 67.941, "idf1": 79.985},
        "mot17_th02": {"hota": 65.878, "idf1": 78.832},
    },
    "MOT17-10": {
        "no_reid": {"hota": 57.204, "idf1": 76.157},
        "mot17_th02": {"hota": 59.565, "idf1": 81.087},
    },
    "MOT17-11": {
        "no_reid": {"hota": 66.697, "idf1": 77.326},
        "mot17_th02": {"hota": 66.699, "idf1": 77.326},
    },
    "MOT17-13": {
        "no_reid": {"hota": 69.833, "idf1": 89.533},
        "mot17_th02": {"hota": 69.791, "idf1": 89.431},
    },
}


def ref_seq_key(seq: str) -> str:
    parts = seq.split("-")
    return f"{parts[0]}-{parts[1]}"


for seq in ACTIVE_SEQUENCES:
    key = ref_seq_key(seq)
    ref = REID_STUDY_PER_SEQ.get(key, {})
    print(seq)
    print(f"  {'Config':<28}  {'HOTA':>6}  {'IDF1':>6}  {'IDSW':>5}  {'Ref H':>6}  {'dH':>6}  {'Ref I':>6}  {'dI':>6}")
    for label, res in botsort_rows:
        hota, assa, idf1, idsw = seq_metrics(res, seq)
        ref_key = "no_reid" if "baseline" in label else "mot17_th02"
        ref_vals = ref.get(ref_key, {})
        ref_hota = ref_vals.get("hota", float("nan"))
        ref_idf1 = ref_vals.get("idf1", float("nan"))
        delta_h = hota - ref_hota if ref_hota == ref_hota else float("nan")
        delta_i = idf1 - ref_idf1 if ref_idf1 == ref_idf1 else float("nan")
        ref_h_s = f"{ref_hota:6.2f}" if ref_hota == ref_hota else "   n/a"
        ref_i_s = f"{ref_idf1:6.2f}" if ref_idf1 == ref_idf1 else "   n/a"
        delta_h_s = f"{delta_h:+6.2f}" if delta_h == delta_h else "   n/a"
        delta_i_s = f"{delta_i:+6.2f}" if delta_i == delta_i else "   n/a"
        print(f"  {label:<28}  {hota:6.2f}  {idf1:6.2f}  {idsw:5d}  {ref_h_s}  {delta_h_s}  {ref_i_s}  {delta_i_s}")
    print()

## Where ReID prevents an ID switch

On the sequence with the largest HOTA gain, find frames where the baseline tracker
switches the ID of a GT pedestrian, while +ReID keeps a consistent ID. Plot a few
of those switch moments side by side.


In [ ]:
COMPARE_SEQ: str | None = None  # override with e.g. "MOT17-02-FRCNN"
MAX_SWITCH_EVENTS = 6
MIN_MATCH_IOU = 0.5


def pedestrian_detections(gt_frame) -> sv.Detections:
    """Eval pedestrians: conf > 0 and class == 1."""
    keep = (gt_frame.confidences > 0) & (gt_frame.classes == 1)
    if not np.any(keep):
        return sv.Detections.empty()
    return sv.Detections(
        xyxy=sv.xywh_to_xyxy(gt_frame.boxes[keep]).astype(np.float32),
        tracker_id=gt_frame.ids[keep].astype(int),
    )


def detections_from_mot(mot: dict, frame_idx: int) -> sv.Detections:
    frame = mot.get(frame_idx)
    if frame is None:
        return sv.Detections.empty()
    active = frame.ids >= 0
    if not np.any(active):
        return sv.Detections.empty()
    return sv.Detections(
        xyxy=sv.xywh_to_xyxy(frame.boxes[active]).astype(np.float32),
        tracker_id=frame.ids[active].astype(int),
        confidence=frame.confidences[active].astype(np.float32),
    )


def match_gt_to_tracks(
    gt_dets: sv.Detections,
    track_dets: sv.Detections,
    *,
    min_iou: float = 0.5,
) -> dict[int, tuple[int, np.ndarray]]:
    """Greedy IoU match from GT ids to tracker ids."""
    if len(gt_dets) == 0 or len(track_dets) == 0:
        return {}
    ious = box_iou(gt_dets.xyxy.astype(np.float64), track_dets.xyxy.astype(np.float64))
    matched_tracks: set[int] = set()
    out: dict[int, tuple[int, np.ndarray]] = {}
    pairs = [
        (float(ious[g, t]), int(g), int(t))
        for g in range(len(gt_dets))
        for t in range(len(track_dets))
        if float(ious[g, t]) >= min_iou
    ]
    pairs.sort(reverse=True)
    for _, g, t in pairs:
        gt_id = int(gt_dets.tracker_id[g])
        if gt_id in out or t in matched_tracks:
            continue
        out[gt_id] = (int(track_dets.tracker_id[t]), gt_dets.xyxy[g].copy())
        matched_tracks.add(t)
    return out


def annotate_tracks(frame_bgr: np.ndarray, detections: sv.Detections, title: str) -> np.ndarray:
    scene = frame_bgr.copy()
    if len(detections) > 0:
        palette, lookup = sv.ColorPalette.DEFAULT, sv.ColorLookup.TRACK
        scene = sv.BoxAnnotator(color=palette, color_lookup=lookup, thickness=2).annotate(scene, detections)
        labels = [str(int(tid)) for tid in detections.tracker_id]
        scene = sv.LabelAnnotator(
            color=palette,
            color_lookup=lookup,
            text_color=sv.Color.BLACK,
            text_scale=0.5,
        ).annotate(scene, detections, labels=labels)
    cv2.putText(
        scene,
        title,
        (12, 32),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )
    return scene


def find_reid_helped_id_switches(
    gt_by_frame: dict,
    mot_base: dict,
    mot_reid: dict,
    n_frames: int,
    *,
    min_iou: float = 0.5,
) -> list[tuple[int, int, int, int, int, np.ndarray]]:
    """Frames where baseline switches GT identity and ReID does not.

    Returns (frame, gt_id, baseline_prev_id, baseline_new_id, reid_id, gt_xyxy).
    """
    prev_base: dict[int, int] = {}
    prev_reid: dict[int, int] = {}
    events: list[tuple[int, int, int, int, int, np.ndarray]] = []

    for frame_idx in range(1, n_frames + 1):
        gt_frame = gt_by_frame.get(frame_idx)
        if gt_frame is None:
            continue
        gt_dets = pedestrian_detections(gt_frame)
        base_map = match_gt_to_tracks(
            gt_dets, detections_from_mot(mot_base, frame_idx), min_iou=min_iou
        )
        reid_map = match_gt_to_tracks(
            gt_dets, detections_from_mot(mot_reid, frame_idx), min_iou=min_iou
        )

        for gt_id, (tid_b, xyxy) in base_map.items():
            if gt_id in prev_base and prev_base[gt_id] != tid_b:
                reid_match = reid_map.get(gt_id)
                if reid_match is None:
                    continue
                tid_r = reid_match[0]
                reid_switched = gt_id in prev_reid and prev_reid[gt_id] != tid_r
                if not reid_switched:
                    events.append((frame_idx, gt_id, prev_base[gt_id], tid_b, tid_r, xyxy))

        for gt_id, (tid_b, _) in base_map.items():
            prev_base[gt_id] = tid_b
        for gt_id, (tid_r, _) in reid_map.items():
            prev_reid[gt_id] = tid_r

    return events


def _draw_focus(frame_bgr: np.ndarray, xyxy: np.ndarray, color=(0, 255, 255)) -> np.ndarray:
    out = frame_bgr.copy()
    x1, y1, x2, y2 = xyxy.astype(int)
    cv2.rectangle(out, (x1, y1), (x2, y2), color, 3)
    return out


seq_gains: list[tuple[str, float, float]] = []
for seq in ACTIVE_SEQUENCES:
    h_b, _, i_b, _ = seq_metrics(result_baseline, seq)
    h_r, _, i_r, _ = seq_metrics(result_reid, seq)
    if h_b == h_b and h_r == h_r:
        seq_gains.append((seq, h_r - h_b, i_r - i_b))

if not seq_gains:
    raise RuntimeError("No per-sequence metrics - run tracking and results cells first.")

seq_gains.sort(key=lambda row: row[1], reverse=True)
print("Per-sequence ReID delta HOTA (largest first):")
for seq, dh, di in seq_gains:
    print(f"  {seq:<20}  delta HOTA {dh:+6.2f}  delta IDF1 {di:+6.2f}")

COMPARE_SEQ = COMPARE_SEQ or seq_gains[0][0]
print(f"\nLooking for ReID-helped ID switches on {COMPARE_SEQ}")

pred_base = OUTPUT_ROOT / "botsort_baseline" / "preds" / f"{COMPARE_SEQ}.txt"
pred_reid = OUTPUT_ROOT / "botsort_reid" / "preds" / f"{COMPARE_SEQ}.txt"
mot_base = load_mot_file(pred_base)
mot_reid = load_mot_file(pred_reid)
gt_by_frame = load_mot_file(SEQUENCE_PATHS[COMPARE_SEQ]["gt"])
img_dir = SEQUENCE_PATHS[COMPARE_SEQ]["img"]
n_frames = SEQUENCE_PATHS[COMPARE_SEQ]["n_frames"]

switch_events = find_reid_helped_id_switches(
    gt_by_frame, mot_base, mot_reid, n_frames, min_iou=MIN_MATCH_IOU
)
print(f"{COMPARE_SEQ}: found {len(switch_events)} baseline ID switches that ReID avoided")
for frame_idx, gt_id, prev_id, new_id, reid_id, _ in switch_events[:MAX_SWITCH_EVENTS]:
    print(
        f"  frame {frame_idx:>4}  GT {gt_id}: "
        f"baseline {prev_id}->{new_id}, ReID stayed {reid_id}"
    )

if not switch_events:
    print("No clear ReID-helped ID switches under the current matching rules.")
else:
    show = switch_events[:MAX_SWITCH_EVENTS]
    fig, axes = plt.subplots(len(show), 2, figsize=(10, 3.2 * len(show)))
    if len(show) == 1:
        axes = np.array([axes])

    for row, (frame_idx, gt_id, prev_id, new_id, reid_id, xyxy) in enumerate(show):
        frame = load_mot_frame_image(img_dir, frame_idx)
        left = annotate_tracks(
            _draw_focus(frame, xyxy),
            detections_from_mot(mot_base, frame_idx),
            f"BASELINE  GT {gt_id}: {prev_id}->{new_id}",
        )
        right = annotate_tracks(
            _draw_focus(frame, xyxy),
            detections_from_mot(mot_reid, frame_idx),
            f"+ REID  GT {gt_id}: id {reid_id}",
        )
        axes[row, 0].imshow(left[:, :, ::-1])
        axes[row, 1].imshow(right[:, :, ::-1])
        axes[row, 0].set_title(f"frame {frame_idx} · baseline ID switch")
        axes[row, 1].set_title(f"frame {frame_idx} · ReID consistent")
        axes[row, 0].axis("off")
        axes[row, 1].axis("off")

    fig.suptitle(f"{COMPARE_SEQ}: where ReID prevents an ID switch", y=1.01)
    fig.tight_layout()
    plt.show()


You just evaluated BoT-SORT with and without appearance ReID on MOT17. Nice work!

Trackers makes it easy to mix and match multi-object tracking algorithms with your
favorite detection backends. Appearance association is optional: install
`trackers[reid]`, pass a `reid.ReIDModel`, and supply `frame=` to `update()`.

Ready to go deeper? Explore the [ReID appearance guide](https://trackers.roboflow.com/latest/learn/reid/),
the [`reid` package](https://reid.roboflow.com/latest/), or the Trackers
[documentation](https://trackers.roboflow.com/latest/) and
[GitHub](https://github.com/roboflow/trackers).

Got feedback or ideas? Open an issue on
[GitHub Issues](https://github.com/roboflow/trackers/issues).